# SECOFS Preprocessing with nos-utils

This notebook demonstrates the complete SECOFS (Southeast Coastal Ocean Forecast System)
preprocessing pipeline using the `nos-utils` Python package.

**Pipeline steps:**
1. Configure the run (domain, cycle, time window)
2. Generate GFS atmospheric forcing (sflux_air_1, sflux_rad_1, sflux_prc_1)
3. Generate HRRR atmospheric forcing (sflux_air_2, sflux_rad_2, sflux_prc_2)
4. Generate tidal boundary conditions (bctides.in)
5. Generate river forcing from USGS climatology (vsource.th, msource.th)
6. Generate model configuration (param.nml)
7. Ocean boundary conditions (OBC) — currently Fortran

**Requirements:**
- `nos-utils` package (`pip install -e .`)
- `wgrib2` on PATH (for GRIB2 extraction)
- Access to GFS/HRRR data (WCOSS2 or downloaded)
- SECOFS FIX files (grid, templates, climatology)

In [ ]:
from pathlib import Path
from datetime import datetime

from nos_utils.config import ForcingConfig
from nos_utils.forcing import (
    GFSProcessor,
    HRRRProcessor,
    TidalProcessor,
    RiverClimProcessor,
    ParamNmlProcessor,
)

## Step 0: Configuration

Create a `ForcingConfig` for SECOFS. This sets the domain bounds, cycle info,
nowcast/forecast window, and processing parameters.

In [ ]:
# Production date and cycle
PDY = "20260403"
CYC = 0  # 00Z cycle

# Hotstart time (from previous cycle's restart file)
time_hotstart = datetime(2026, 4, 2, 18, 0, 0)  # 18Z previous day

# Create SECOFS config with defaults
config = ForcingConfig.for_secofs(pdy=PDY, cyc=CYC)

print(f"OFS:           SECOFS")
print(f"PDY:           {config.pdy}")
print(f"Cycle:         {config.cyc:02d}Z")
print(f"Domain:        lon=[{config.lon_min}, {config.lon_max}] lat=[{config.lat_min}, {config.lat_max}]")
print(f"Nowcast:       {config.nowcast_hours}h")
print(f"Forecast:      {config.forecast_hours}h")
print(f"Met sources:   {config.met_num} (1=GFS only, 2=GFS+HRRR)")
print(f"NWS mode:      {config.nws} (2=sflux, 4=DATM)")
print(f"SSH offset:    {config.obc_ssh_offset}m (geoid-to-MSL)")

In [ ]:
# Define input/output paths
# On WCOSS2 these come from environment variables set by the J-job
import os

COMINgfs  = Path(os.environ.get("COMINgfs",  "/lfs/h1/ops/prod/com/gfs/v16.3"))
COMINhrrr = Path(os.environ.get("COMINhrrr", "/lfs/h1/ops/prod/com/hrrr/v4.1"))
FIXofs    = Path(os.environ.get("FIXofs",     "/path/to/fix/secofs"))
DATA      = Path(os.environ.get("DATA",       "/tmp/secofs_prep"))
COMOUT    = Path(os.environ.get("COMOUT",     "/tmp/secofs_comout"))

DATA.mkdir(parents=True, exist_ok=True)
COMOUT.mkdir(parents=True, exist_ok=True)

print(f"COMINgfs:  {COMINgfs}")
print(f"COMINhrrr: {COMINhrrr}")
print(f"FIXofs:    {FIXofs}")
print(f"DATA:      {DATA}")
print(f"COMOUT:    {COMOUT}")

## Step 1: GFS Atmospheric Forcing

Extract GFS 0.50° GRIB2 data for the SECOFS domain and write SCHISM sflux files.

**Input:** GFS GRIB2 files from COMINgfs  
**Output:** `sflux_air_1.{N}.nc`, `sflux_rad_1.{N}.nc`, `sflux_prc_1.{N}.nc`

Variables: uwind, vwind, prmsl, stmp, spfh (air), dlwrf, dswrf (rad), prate (prc)

In [ ]:
# --- GFS Nowcast ---
gfs_now = GFSProcessor(
    config=config,
    input_path=COMINgfs,
    output_path=DATA / "sflux",
    resolution="0p50",
    phase="nowcast",
    time_hotstart=time_hotstart,
)

# Check what files would be used
gfs_files = gfs_now.find_input_files()
print(f"GFS nowcast: found {len(gfs_files)} input files")
for f in gfs_files[:5]:
    print(f"  {f.name}")
if len(gfs_files) > 5:
    print(f"  ... and {len(gfs_files) - 5} more")

In [ ]:
# Process GFS nowcast
result_gfs_now = gfs_now.process()
print(f"Success: {result_gfs_now.success}")
print(f"Output files: {[f.name for f in result_gfs_now.output_files]}")
if result_gfs_now.metadata:
    print(f"Timesteps: {result_gfs_now.metadata.get('num_timesteps')}")
    print(f"Grid: {result_gfs_now.metadata.get('grid_shape')}")

In [ ]:
# --- GFS Forecast ---
# For forecast, uses single GFS cycle with extended leads (matching COMF production)
gfs_fc = GFSProcessor(
    config=config,
    input_path=COMINgfs,
    output_path=DATA / "sflux",
    resolution="0p50",
    phase="forecast",
    time_hotstart=time_hotstart,
)

result_gfs_fc = gfs_fc.process()
print(f"GFS forecast: success={result_gfs_fc.success}, "
      f"files={len(result_gfs_fc.output_files)}, "
      f"timesteps={result_gfs_fc.metadata.get('num_timesteps', 'N/A')}")

## Step 2: HRRR Atmospheric Forcing

Extract HRRR 3km data (regridded to lat/lon) as secondary sflux source.
SCHISM blends GFS (sflux_air_1) with HRRR (sflux_air_2) where HRRR has coverage.

**Input:** HRRR GRIB2 files from COMINhrrr  
**Output:** `sflux_air_2.{N}.nc`, `sflux_rad_2.{N}.nc`, `sflux_prc_2.{N}.nc`

In [ ]:
# --- HRRR Nowcast ---
hrrr_now = HRRRProcessor(
    config=config,
    input_path=COMINhrrr,
    output_path=DATA / "sflux",
    regrid_dx=0.03,  # 0.03° ≈ 3.3km lat/lon grid
    phase="nowcast",
    time_hotstart=time_hotstart,
)

result_hrrr_now = hrrr_now.process()
print(f"HRRR nowcast: success={result_hrrr_now.success}")

# --- HRRR Forecast ---
hrrr_fc = HRRRProcessor(
    config=config,
    input_path=COMINhrrr,
    output_path=DATA / "sflux",
    regrid_dx=0.03,
    phase="forecast",
    time_hotstart=time_hotstart,
)

result_hrrr_fc = hrrr_fc.process()
print(f"HRRR forecast: success={result_hrrr_fc.success}")

## Step 3: Tidal Boundary Conditions

Generate `bctides.in` with tidal constituent amplitudes/phases and nodal corrections.
Uses the Fortran `nos_ofs_create_tide_fac_schism` executable if available (recommended),
otherwise falls back to Python approximate nodal corrections.

**Input:** `bctides.in_template` from FIX directory  
**Output:** `bctides.in` (nowcast and forecast versions)

In [ ]:
# --- Tidal nowcast ---
tidal_now = TidalProcessor(
    config=config,
    input_path=FIXofs,
    output_path=DATA,
    phase="nowcast",
    time_hotstart=time_hotstart,
)

result_tidal = tidal_now.process()
print(f"Tidal: success={result_tidal.success}")
print(f"Mode: {result_tidal.metadata.get('mode', 'unknown')}")
# Mode will be:
#   'fortran_tide_fac' — used Fortran exe (accurate, production)
#   'template'         — Python nodal corrections (approximate)
#   'copy'             — static bctides.in from FIX
#   'python_native'    — generated from scratch (minimal)

## Step 4: River Forcing

SECOFS has 6 river input nodes (Fraser, Columbia, Willamette, Clackamas, Vancouver, Lewis).
NWM has no matching reaches for this domain, so forcing comes from USGS daily climatology.

The `RiverClimProcessor` reads:
- `secofs.river.ctl` — node mappings and scaling factors
- `nosofs.river.clim.usgs.nc` — daily climatological discharge, temperature, salinity

**Output:** `vsource.th`, `msource.th`, `source_sink.in`

In [ ]:
river = RiverClimProcessor(
    config=config,
    input_path=FIXofs,
    output_path=DATA,
    phase="nowcast",
    time_hotstart=time_hotstart,
)

result_river = river.process()
print(f"River: success={result_river.success}")
print(f"Output: {[f.name for f in result_river.output_files]}")
if result_river.metadata:
    print(f"Nodes: {result_river.metadata.get('n_nodes')}")
    print(f"Clim station: {result_river.metadata.get('clim_station')}")

## Step 5: Model Configuration (param.nml)

Generate SCHISM `param.nml` from a template, substituting runtime values:
- `rnday`: Run duration in days
- `start_year/month/day/hour`: Model start time
- `ihot`: Hotstart mode (1=nowcast, 2=forecast)
- `nws`: Atmospheric forcing mode

In [ ]:
# --- param.nml for nowcast ---
param_now = ParamNmlProcessor(
    config=config,
    input_path=FIXofs,
    output_path=DATA,
    phase="nowcast",
    time_hotstart=time_hotstart,
)

result_param = param_now.process()
print(f"param.nml (nowcast): success={result_param.success}")
if result_param.metadata:
    print(f"  rnday={result_param.metadata.get('rnday')}")
    print(f"  ihot={result_param.metadata.get('ihot')}")
    print(f"  start={result_param.metadata.get('start_time')}")

## Step 6: Ocean Boundary Conditions (OBC)

OBC generation still uses the Fortran executable `nos_ofs_create_forcing_obc_schism`.
This produces:
- `elev2D.th.nc` — SSH boundary
- `TEM_3D.th.nc` — Temperature boundary
- `SAL_3D.th.nc` — Salinity boundary
- `uv3D.th.nc` — Velocity boundary
- `TEM_nu.nc`, `SAL_nu.nc` — Interior nudging

The Python `RTOFSProcessor` exists in nos-utils but is not yet validated
for production. The Fortran executable is called via subprocess.

In [ ]:
# OBC is still handled by Fortran — shown here for documentation
# In the shell workflow, this is called by nos_ofs_prep_run.sh:
#   $EXECnos/nos_ofs_create_forcing_obc_schism < Fortran_OBC.ctl > Fortran_OBC.log
print("OBC: Generated by Fortran nos_ofs_create_forcing_obc_schism")
print("  Outputs: elev2D.th.nc, TEM_3D.th.nc, SAL_3D.th.nc, uv3D.th.nc")
print("  Nudging: TEM_nu.nc, SAL_nu.nc")

## Summary: Pipeline Status

| Step | Processor | Generator | Status |
|------|-----------|-----------|--------|
| GFS sflux | `GFSProcessor` | nos-utils | Validated (R=1.0 vs COMF) |
| HRRR sflux | `HRRRProcessor` | nos-utils | Validated (bit-for-bit vs COMF) |
| bctides.in | `TidalProcessor` | Fortran exe via nos-utils | Validated (exact match) |
| River forcing | `RiverClimProcessor` | nos-utils | Validated (exact match) |
| param.nml | `ParamNmlProcessor` | nos-utils | Validated (formatting-only diff) |
| OBC | Fortran exe | Legacy Fortran | Bit-for-bit (same exe) |
| Nudging | Fortran exe | Legacy Fortran | Bit-for-bit (same exe) |

## Inspecting Output

Quick visualization of the generated sflux files.

In [ ]:
# Inspect a generated sflux file
from netCDF4 import Dataset
import numpy as np

sflux_dir = DATA / "sflux"
air_files = sorted(sflux_dir.glob("sflux_air_1.*.nc"))

if air_files:
    ds = Dataset(str(air_files[0]))
    print(f"File: {air_files[0].name}")
    print(f"Dimensions: {dict(ds.dimensions)}")
    print(f"Variables: {list(ds.variables.keys())}")
    print(f"Time: {np.array(ds.variables['time'][:])}")
    print(f"Base date: {np.array(ds.variables['time'].base_date)}")
    print(f"Grid shape: lon={ds.variables['lon'].shape}, lat={ds.variables['lat'].shape}")
    
    # Show wind speed range
    u = np.array(ds.variables['uwind'][0])
    v = np.array(ds.variables['vwind'][0])
    ws = np.sqrt(u**2 + v**2)
    print(f"Wind speed at t=0: [{ws.min():.1f}, {ws.max():.1f}] m/s")
    ds.close()
else:
    print("No sflux files found — run the GFS processor first")

In [ ]:
# Quick spatial plot of wind speed
import matplotlib.pyplot as plt

if air_files:
    ds = Dataset(str(air_files[0]))
    lon = np.array(ds.variables['lon'][:])
    lat = np.array(ds.variables['lat'][:])
    u = np.array(ds.variables['uwind'][0])
    v = np.array(ds.variables['vwind'][0])
    ws = np.sqrt(u**2 + v**2)
    ds.close()

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.pcolormesh(lon, lat, ws, cmap='YlOrRd', shading='auto')
    plt.colorbar(im, label='Wind Speed (m/s)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'SECOFS GFS Wind Speed — {air_files[0].name}')
    plt.show()

In [ ]:
# Inspect river forcing
vsource_file = DATA / "vsource.th"
if vsource_file.exists():
    vs = np.loadtxt(vsource_file)
    print(f"vsource.th: {vs.shape[0]} timesteps, {vs.shape[1]-1} rivers")
    print(f"Time range: {vs[0,0]:.0f}s to {vs[-1,0]:.0f}s ({vs[-1,0]/3600:.0f} hours)")
    print(f"Q per node: {vs[0,1]:.4f} m³/s (constant climatological value)")
    print(f"Total Q: {vs[0,1:].sum():.1f} m³/s")
else:
    print("vsource.th not found — run the RiverClimProcessor first")